# **📄 Clean & Jerk Dataset — Publication Figures & Tables (`Plan.md`)**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    build_annotation_agreement_table,
    build_imu_quality_table,
    build_paper_tables,
    plot_fig4_annotation_scheme,
    plot_all_sets_annotation_sd_heatmap,
    plot_annotation_agreement_figure,
    plot_imu_validation_figure,
    plot_set_annotator_grid,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 📊 Paper Tables

In [ ]:
# @title 📊 Generate Paper Tables 5, 7, 8
paper_tables = build_paper_tables(dataset_dir)
for name, df in paper_tables.items():
    print(f"\n=== {name} ===")
    display_table(df, clean_columns=False)


=== table5_demographics ===


,participant_id,name,age_years,sex,height_m,body_mass_kg,bmi_kg_m2,experience_years,achievement,best_weight_kg,notes,sets,lifts
0,P0001,Enuka,NaN,M,NaN,NaN,NaN,0.00,NaN,NaN,NaN,2,10
1,P0002,Thevindu Samarasekera,25.0,M,1.69,62.0,21.71,0.25,NaN,55.0,"Participation in Other sports: Swimming, Badmi...",4,18
2,P0003,Achalani Royal,23.0,F,1.60,49.0,19.14,1.50,Central Province Festival Selection 4th place,55.0,Participation in Other sports: None,2,10
3,P0004,Pandula,NaN,M,NaN,NaN,NaN,0.00,NaN,NaN,NaN,2,10
4,P0005,Induwara Liyanage,21.0,M,1.70,62.0,21.45,1.00,NaN,50.0,Participation in Other sports: Swimming,2,10
5,P0006,Duleetha,NaN,M,NaN,NaN,NaN,0.00,NaN,NaN,NaN,2,10
6,P0007,Hivindu Kirulapana,22.0,M,1.74,72.0,23.78,1.00,NaN,NaN,Participation in Other sports: Rugby,2,10
7,P0008,Yewin Dissanayake,22.0,M,1.70,65.0,22.49,2.00,Intermediate bronze medal 2025,95.0,Participation in Other sports: None,3,13
8,P0009,Yasith Dissanayake,22.0,M,1.78,69.0,21.78,4.00,NaN,55.0,Participation in Other sports: Rugby,3,13
9,P0010,J W K Mahasen Jayasundara,24.0,M,1.78,69.0,21.78,5.00,NaN,50.0,"Participation in Other sports: Rugby, Badminton",4,16



=== table7_coverage ===


,tier,sets,participants,lifts_total,lifts_usable,duration_min,total_imu_samples
0,Complete (No Faults),35,15,157,157,44.37,266229
1,Fault-Tolerant,19,9,89,80,25.36,152162
2,Total,54,19,246,237,69.73,418391



=== table8_agreement ===


,marker_name,n_boundaries,mean_sd_frames,mean_sd_ms,max_sd_frames,pct_within_6_frames,pct_within_8_frames
0,start,246,2.64,88.0,7.51,90.7,100.0
1,setup,246,2.79,93.2,7.21,95.5,100.0
2,first_pull,246,2.13,70.9,7.51,98.4,100.0
3,second_pull,246,1.51,50.4,5.51,100.0,100.0
4,clean,246,2.27,75.5,5.69,100.0,100.0
5,intermediate_rest,246,3.11,103.5,7.55,89.4,100.0
6,jerk_split,246,2.27,75.7,7.09,97.6,100.0
7,jerk_recovery,246,2.85,95.1,7.51,91.9,100.0
8,overhead_hold,246,3.37,112.3,8.00,88.2,100.0
9,descend,246,2.51,83.8,6.43,97.2,100.0


### 🖼️ 11-Phase Annotation Scheme

In [ ]:
# @title 🖼️ Figure 4: Annotation Scheme { display-mode: "form" }
sample_set_id = "set_0054" # @param {type:"string"}
sample_lifts = "2" # @param {type:"string"}  # e.g. "1", "1,2", "1-5", or "all"
sample_sensors = "LHD" # @param {type:"string"}  # e.g. "RHD" or "RKP,RHD" or "US,LS"
modality = "both" # @param ["gyro", "accel", "both"]
signal_mode = "axes" # @param ["axes", "magnitude", "both"]
axes_str = "X,Y,Z" # @param {type:"string"}  # e.g. "X,Y,Z" or "Y" or "X,Z"

fig4 = plot_fig4_annotation_scheme(
    dataset_dir,
    set_id=sample_set_id,
    lift_ids=sample_lifts,
    sensor_codes=sample_sensors,
    modality=modality,
    signal_mode=signal_mode,
    axes=axes_str,
)
fig4.show(config=EXPORT_CFG)

### 📊 IMU Technical Validation Summary Table

In [ ]:
# @title 📊 IMU Quality Summary Table (Option 1)
df_imu_quality = build_imu_quality_table(dataset_dir)
display_table(df_imu_quality, clean_columns=False)

📝 Draft text: "Across all 1,404 sensor recordings (54 sets × 26 IMUs), data completeness was 97.4 ± 15.8% (min 0.0%). The accelerometer norm during static periods was 9.67 ± 1.27 m/s², and static gyroscope bias was 10.29 ± 8.87 deg/s."


,Metric,Expected / Threshold,Mean ± SD,Median [IQR],Min – Max,Recordings Passing (%)
0,Data completeness (% valid samples),> 99%,97.43 ± 15.81,"100.00 [100.00, 100.00]",0.00 – 100.00,97.4%
1,Missing / flatlined samples (%),< 1%,2.57 ± 15.81,"0.00 [0.00, 0.00]",0.00 – 100.00,97.4%
2,Static accelerometer norm (m/s²),9.81 ± 0.50 m/s²,9.67 ± 1.27,"9.81 [9.62, 10.06]",0.00 – 12.39,87.7%
3,Static gyroscope bias magnitude (deg/s),< 5.0 deg/s,10.29 ± 8.87,"7.90 [5.35, 12.13]",0.00 – 125.50,20.8%
4,Accelerometer saturation (% > 80 m/s²),≈ 0% (< 0.1%),0.00 ± 0.00,"0.00 [0.00, 0.00]",0.00 – 0.00,100.0%
5,Gyroscope saturation (% > 2000 deg/s),≈ 0% (< 0.1%),0.00 ± 0.00,"0.00 [0.00, 0.00]",0.00 – 0.00,100.0%
6,Inter-IMU dynamic lag vs. US (ms),< 10 ms (1 sample),104.64 ± 73.98,"100.00 [30.00, 190.00]",0.00 – 200.00,16.1%


### 🖼️ IMU Technical Validation 4-Panel Figure

In [ ]:
# @title 🖼️ IMU Technical Validation Figure (Option 2)
fig_val = plot_imu_validation_figure(dataset_dir)
fig_val.show(config=EXPORT_CFG)

### 🔥 Per-Set Annotator Grid & SD (ms) Heatmap

In [ ]:
# @title 🔥 Per-Set Annotator Grid { display-mode: "form" }
ann_set_id = "set_0001" # @param {type:"string"}
fig_ann_set = plot_set_annotator_grid(dataset_dir, set_id=ann_set_id)
fig_ann_set.show(config=EXPORT_CFG)

### 🔥 All-Sets Inter-Annotator SD (ms) Heatmap

In [ ]:
# @title 🔥 All-Sets Inter-Annotator SD (ms) Heatmap
fig_ann_all = plot_all_sets_annotation_sd_heatmap(dataset_dir)
fig_ann_all.show(config=EXPORT_CFG)

### 📊 Annotation Agreement Summary Table (Right-Skewed)

In [ ]:
# @title 📊 Annotation Agreement Table { display-mode: "form" }
threshold_ms = 200.0 # @param {type:"number"}  # 200 ms = 6 frames @ 30 fps (20 samples @ 100 Hz)
df_ann_table = build_annotation_agreement_table(dataset_dir, threshold_ms=threshold_ms)
display_table(df_ann_table, clean_columns=False)

### 🖼️ Annotation Agreement 2-Panel Figure (Histogram + ECDF)

In [ ]:
# @title 🖼️ Annotation Agreement (Histogram + ECDF) { display-mode: "form" }
log_x_ecdf = False # @param {type:"boolean"}
fig_ann_agree = plot_annotation_agreement_figure(dataset_dir, log_x_ecdf=log_x_ecdf)
fig_ann_agree.show(config=EXPORT_CFG)